# Baseline HP Prediction Model

## Three-Phase Approach:
1. **Phase 1**: Get baseline HP estimate based on CR from Lazy 5e Monster Stats
2. **Phase 2**: Calculate HP penalties for deviations from CR baselines (AC, Attack Bonus, DPR)
3. **Phase 3**: Calculate HP adjustments for special abilities (conditions, resistances, etc.)

**Key Changes from Previous Model:**
- Uses CR-specific baselines for AC, Attack, and DPR
- Penalties apply based on deviation from baseline, not absolute values
- No monster type features (removed from feature engineering)
- More accurate for low-CR creatures like Aarakocra

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Imports successful")

✅ Imports successful


## Load Data

In [ ]:
# Load monster dataset
df = pd.read_csv('./data/dnd5e_monsters_2014.csv')
df = pd.read_csv('./')
print(f"📊 Loaded {len(df)} monsters from dataset")

# Load Lazy 5e baseline stats
lazy_5e = pd.read_csv('../data/lazy_5e_monster_stats_by_cr.csv')
print(f"📊 Loaded {len(lazy_5e)} CR baselines from Lazy 5e")

print("\n🔍 Lazy 5e Baseline Stats Preview:")
display(lazy_5e.head(10))

📊 Loaded 324 monsters from dataset


FileNotFoundError: [Errno 2] No such file or directory: '../data/lazy_5e_monster_stats_by_cr.csv'

## Parse CR and Create Baseline Lookup Functions

In [ ]:
# Parse CR from string to numeric
def parse_cr(cr_str):
    if pd.isna(cr_str):
        return 0
    cr_str = str(cr_str).strip()
    if '/' in cr_str:
        num, denom = cr_str.split('/')
        return float(num) / float(denom)
    try:
        return float(cr_str)
    except:
        return 0

# Convert Lazy 5e CR to numeric
lazy_5e['cr_numeric'] = lazy_5e['CR'].apply(parse_cr)

# Parse HP from Lazy 5e (extract average from "65 (49-81)" format)
def parse_hp_avg(hp_str):
    if pd.isna(hp_str):
        return 0
    match = re.match(r'(\d+)', str(hp_str))
    return int(match.group(1)) if match else 0

lazy_5e['hp_baseline'] = lazy_5e['HP'].apply(parse_hp_avg)

# Adjust HP baselines: +50% for CR <= 1, +20% for CR >= 2
def adjust_hp_baseline(row):
    if row['cr_numeric'] <= 1.0:
        return row['hp_baseline'] * 1.5
    else:
        return row['hp_baseline'] * 1.2

lazy_5e['hp_baseline'] = lazy_5e.apply(adjust_hp_baseline, axis=1)
print("✅ Adjusted HP baselines: +50% for CR ≤ 1, +20% for CR ≥ 2")

# Parse Attack Bonus - handles both numeric (3) and "+3" format
def parse_bonus(bonus_str):
    if pd.isna(bonus_str):
        return 0
    bonus_str = str(bonus_str).strip()
    # Try to match "+3" format first
    match = re.search(r'\+(\d+)', bonus_str)
    if match:
        return int(match.group(1))
    # Try direct numeric format
    try:
        return int(bonus_str)
    except:
        return 0

lazy_5e['attack_baseline'] = lazy_5e['Attack_Bonus'].apply(parse_bonus)
lazy_5e['ac_baseline'] = lazy_5e['AC_DC']
lazy_5e['dpr_baseline'] = lazy_5e['Damage_Round']

print("✅ Parsed Lazy 5e baselines")
print("\n📊 Baseline Stats by CR:")
display(lazy_5e[['cr_numeric', 'hp_baseline', 'ac_baseline', 'attack_baseline', 'dpr_baseline']].head(15))

In [ ]:
# Create interpolation functions for baselines
# This allows us to get baseline values for any CR, even fractional ones

cr_values = lazy_5e['cr_numeric'].values
hp_baseline_interp = interp1d(cr_values, lazy_5e['hp_baseline'].values, 
                               kind='linear', bounds_error=False, fill_value='extrapolate')
ac_baseline_interp = interp1d(cr_values, lazy_5e['ac_baseline'].values,
                               kind='linear', bounds_error=False, fill_value='extrapolate')
attack_baseline_interp = interp1d(cr_values, lazy_5e['attack_baseline'].values,
                                   kind='linear', bounds_error=False, fill_value='extrapolate')
dpr_baseline_interp = interp1d(cr_values, lazy_5e['dpr_baseline'].values,
                                kind='linear', bounds_error=False, fill_value='extrapolate')

def get_baseline_hp(cr):
    return float(hp_baseline_interp(cr))

def get_baseline_ac(cr):
    return float(ac_baseline_interp(cr))

def get_baseline_attack(cr):
    return float(attack_baseline_interp(cr))

def get_baseline_dpr(cr):
    return float(dpr_baseline_interp(cr))

print("✅ Created baseline interpolation functions")
print("\n🧪 Test baselines for CR 0.25 (Aarakocra):")
test_cr = 0.25
print(f"  HP Baseline: {get_baseline_hp(test_cr):.1f}")
print(f"  AC Baseline: {get_baseline_ac(test_cr):.1f}")
print(f"  Attack Baseline: {get_baseline_attack(test_cr):.1f}")
print(f"  DPR Baseline: {get_baseline_dpr(test_cr):.1f}")

## Feature Engineering (No Monster Types)

In [ ]:
print("⚙️  FEATURE ENGINEERING")
print("=" * 60)

# Parse CR
df['cr_numeric'] = df['Challenge_Rating'].apply(parse_cr)

# Parse AC
def parse_ac(ac_str):
    if pd.isna(ac_str):
        return 10
    match = re.search(r'\d+', str(ac_str))
    return int(match.group()) if match else 10

df['ac_value'] = df['AC'].apply(parse_ac)

# Parse speeds
def parse_speed(speed_str, speed_type):
    if pd.isna(speed_str):
        return 0
    speed_str = str(speed_str).lower()
    if speed_type == 'ground':
        match = re.search(r'^(\d+)\s*ft', speed_str)
        return int(match.group(1)) if match else 0
    else:
        pattern = rf'{speed_type}\s+(\d+)\s*ft'
        match = re.search(pattern, speed_str)
        return int(match.group(1)) if match else 0

df['speed_ground'] = df['Speed'].apply(lambda x: parse_speed(x, 'ground'))
df['speed_fly'] = df['Speed'].apply(lambda x: parse_speed(x, 'fly'))
df['speed_swim'] = df['Speed'].apply(lambda x: parse_speed(x, 'swim'))
df['speed_burrow'] = df['Speed'].apply(lambda x: parse_speed(x, 'burrow'))
df['speed_climb'] = df['Speed'].apply(lambda x: parse_speed(x, 'climb'))

df['max_speed'] = df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']].max(axis=1)
df['movement_types_count'] = (df[['speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb']] > 0).sum(axis=1)
df['has_flying'] = (df['speed_fly'] > 0).astype(int)

# Parse proficiencies
def count_proficiencies(prof_str):
    if pd.isna(prof_str) or str(prof_str).strip() == '':
        return 0
    return len([x.strip() for x in str(prof_str).split(',') if x.strip()])

df['save_proficiency_count'] = df['Saving_Throws'].apply(count_proficiencies)
df['skill_proficiency_count'] = df['Skills'].apply(count_proficiencies)
df['resistance_count'] = df['Resistances'].apply(count_proficiencies)
df['immunity_count'] = df['Immunities'].apply(count_proficiencies)
df['vulnerability_count'] = df['Vulnerabilities'].apply(count_proficiencies)
df['condition_immunity_count'] = df['Condition_Immunities'].apply(count_proficiencies)

print("✅ Basic features parsed")

In [ ]:
# Parse senses
def has_sense(sense_str, sense_type):
    if pd.isna(sense_str):
        return 0
    return 1 if sense_type in str(sense_str).lower() else 0

def parse_sense_range(sense_str, sense_type):
    if pd.isna(sense_str):
        return 0
    pattern = rf'{sense_type}\s+(\d+)\s*ft'
    match = re.search(pattern, str(sense_str).lower())
    return int(match.group(1)) if match else 0

df['has_darkvision'] = df['Senses'].apply(lambda x: has_sense(x, 'darkvision'))
df['darkvision_range'] = df['Senses'].apply(lambda x: parse_sense_range(x, 'darkvision'))
df['has_blindsight'] = df['Senses'].apply(lambda x: has_sense(x, 'blindsight'))
df['has_truesight'] = df['Senses'].apply(lambda x: has_sense(x, 'truesight'))
df['has_tremorsense'] = df['Senses'].apply(lambda x: has_sense(x, 'tremorsense'))

def parse_passive_perception(sense_str):
    if pd.isna(sense_str):
        return 10
    match = re.search(r'passive\s+perception\s+(\d+)', str(sense_str).lower())
    return int(match.group(1)) if match else 10

df['passive_perception'] = df['Senses'].apply(parse_passive_perception)

print("✅ Senses parsed")

In [ ]:
# Parse abilities from text
def count_abilities(text_str):
    if pd.isna(text_str) or str(text_str).strip() == '':
        return 0
    text = str(text_str).strip()
    entries = [x for x in re.split(r'\n+|\*\s+', text) if x.strip()]
    return len(entries)

df['trait_count'] = df['Traits'].apply(count_abilities)
df['action_count'] = df['Actions'].apply(count_abilities)
df['reaction_count'] = df['Reactions'].apply(count_abilities)
df['bonus_action_count'] = df['Bonus_Actions'].apply(count_abilities)

# Legendary actions
def parse_legendary_actions(leg_str):
    if pd.isna(leg_str) or str(leg_str).strip() == '':
        return 0, 0, 0
    text = str(leg_str).lower()
    has_leg = 1
    count_match = re.search(r'(\d+)\s+legendary\s+actions?', text)
    per_round = int(count_match.group(1)) if count_match else 3
    action_count = len([x for x in re.split(r'\n+|\*\s+', text) if x.strip() and 'can take' not in x.lower()])
    return has_leg, action_count, per_round

df[['has_legendary_actions', 'legendary_action_count', 'legendary_actions_per_round']] = df['Legendary_Actions'].apply(
    lambda x: pd.Series(parse_legendary_actions(x))
)

df['total_ability_count'] = df['trait_count'] + df['action_count'] + df['reaction_count'] + df['legendary_action_count']

print("✅ Ability counts parsed")

In [ ]:
# Parse specific abilities
combined_abilities = (df['Traits'].fillna('') + ' ' + df['Actions'].fillna('') + ' ' +
                     df['Reactions'].fillna('') + ' ' + df['Legendary_Actions'].fillna(''))

df['has_multiattack'] = combined_abilities.str.contains('multiattack', case=False, na=False).astype(int)

# Combat metrics
def parse_attack_bonus(actions_str):
    if pd.isna(actions_str):
        return 0
    matches = re.findall(r'\+(\d+)\s+to\s+hit', str(actions_str).lower())
    return max([int(m) for m in matches]) if matches else 0

def parse_save_dc(text_str):
    if pd.isna(text_str):
        return 0
    matches = re.findall(r'dc\s+(\d+)', str(text_str).lower())
    return max([int(m) for m in matches]) if matches else 0

df['highest_attack_bonus'] = df['Actions'].apply(parse_attack_bonus)
df['highest_save_dc'] = combined_abilities.apply(parse_save_dc)

# Estimate DPR from action text
def parse_dpr(actions_str):
    if pd.isna(actions_str):
        return 0
    actions_str = str(actions_str).lower()
    total_dpr = 0
    damage_patterns = re.findall(r'(\d+)d(\d+)(?:\s*\+\s*(\d+))?', actions_str)
    for num_dice, die_size, modifier in damage_patterns:
        num_dice = int(num_dice)
        die_size = int(die_size)
        modifier = int(modifier) if modifier else 0
        avg_damage = num_dice * (die_size + 1) / 2 + modifier
        total_dpr += avg_damage
    if 'multiattack' in actions_str:
        total_dpr *= 1.5
    return total_dpr

df['estimated_dpr'] = df['Actions'].apply(parse_dpr)

# Special traits
df['has_legendary_resistance'] = combined_abilities.str.contains('legendary resistance', case=False, na=False).astype(int)
df['has_magic_resistance'] = combined_abilities.str.contains('magic resistance', case=False, na=False).astype(int)
df['has_regeneration'] = combined_abilities.str.contains('regeneration', case=False, na=False).astype(int)

# Spellcasting
def parse_spellcasting(text_str):
    if pd.isna(text_str):
        return 0, 0
    text = str(text_str).lower()
    if 'spellcasting' not in text and 'innate spellcasting' not in text:
        return 0, 0
    has_spellcasting = 1
    level_match = re.search(r'(\d+)(?:st|nd|rd|th)[-\s]level\s+spellcaster', text)
    if level_match:
        return has_spellcasting, int(level_match.group(1))
    # Estimate from highest spell level
    for spell_level, caster_level in [(9, 17), (8, 15), (7, 13), (6, 11), (5, 9), (4, 7), (3, 5), (2, 3), (1, 1)]:
        if f'{spell_level}th level' in text or f'{spell_level}st level' in text or f'{spell_level}nd level' in text or f'{spell_level}rd level' in text:
            return has_spellcasting, caster_level
    return has_spellcasting, 0

df[['has_spellcasting', 'spellcaster_level']] = df['Traits'].apply(
    lambda x: pd.Series(parse_spellcasting(x))
)

# Size ordinal
size_map = {'Tiny': 0, 'Small': 1, 'Medium': 2, 'Large': 3, 'Huge': 4, 'Gargantuan': 5}
df['size_ordinal'] = df['Size'].map(size_map).fillna(2)

# Grapple
df['has_grapple'] = combined_abilities.str.contains('grapple', case=False, na=False).astype(int)

print("✅ Special abilities parsed")

In [ ]:
# Condition infliction features
conditions = [
    'poisoned', 'blinded', 'charmed', 'deafened', 'frightened',
    'incapacitated', 'paralyzed', 'petrified', 'prone', 'restrained', 'stunned'
]

for condition in conditions:
    feature_name = f'inflicts_{condition}'
    df[feature_name] = combined_abilities.str.contains(condition, case=False, na=False).astype(int)

print(f"✅ {len(conditions)} condition features added")
print("\n✅ Feature engineering complete (NO monster type features)")

## Phase 1: Calculate Baseline HP and Deviations

In [ ]:
print("📊 PHASE 1: CALCULATING BASELINES AND DEVIATIONS")
print("=" * 60)

# For each monster, get their CR-based baselines
df['hp_baseline'] = df['cr_numeric'].apply(get_baseline_hp)
df['ac_baseline'] = df['cr_numeric'].apply(get_baseline_ac)
df['attack_baseline'] = df['cr_numeric'].apply(get_baseline_attack)
df['dpr_baseline'] = df['cr_numeric'].apply(get_baseline_dpr)

# Calculate deviations from baseline
df['ac_deviation'] = df['ac_value'] - df['ac_baseline']
df['attack_deviation'] = df['highest_attack_bonus'] - df['attack_baseline']
df['dpr_deviation'] = df['estimated_dpr'] - df['dpr_baseline']

print("✅ Baselines and deviations calculated")
print("\n📊 Sample deviations:")
display(df[['Name', 'cr_numeric', 'hp_baseline', 'ac_deviation', 'attack_deviation', 'dpr_deviation']].head(10))

# Create scaled features for powerful abilities
# These scale with baseline HP to avoid disproportionate impact on low-CR creatures
print("\n📐 Creating scaled ability features...")
df['has_flying_scaled'] = df['has_flying'] * df['hp_baseline']
df['has_legendary_resistance_scaled'] = df['has_legendary_resistance'] * df['hp_baseline']
df['has_magic_resistance_scaled'] = df['has_magic_resistance'] * df['hp_baseline']
df['has_regeneration_scaled'] = df['has_regeneration'] * df['hp_baseline']
df['has_legendary_actions_scaled'] = df['has_legendary_actions'] * df['hp_baseline']

print(f"✅ Created 5 scaled ability features")
print(f"   - has_flying_scaled")
print(f"   - has_legendary_resistance_scaled")
print(f"   - has_magic_resistance_scaled")
print(f"   - has_regeneration_scaled")
print(f"   - has_legendary_actions_scaled")


## Build Feature Matrix

In [ ]:
print("🔧 BUILDING FEATURE MATRIX")
print("=" * 60)

# Phase 1 features: CR baseline
phase1_features = ['hp_baseline']

# Phase 2 features: Deviations and scaled abilities (with fixed penalties)
phase2_features = [
    'ac_deviation', 
    'attack_deviation', 
    'dpr_deviation',
    'has_flying_scaled',
    'has_legendary_resistance_scaled',
    'has_magic_resistance_scaled',
    'has_regeneration_scaled',
    'has_legendary_actions_scaled'
]

# Phase 3 features: Other abilities (learned coefficients)
phase3_features = [
    'speed_ground', 'speed_fly', 'speed_swim', 'speed_burrow', 'speed_climb',
    'max_speed', 'movement_types_count',
    'save_proficiency_count', 'skill_proficiency_count',
    'resistance_count', 'immunity_count', 'vulnerability_count', 'condition_immunity_count',
    'has_darkvision', 'darkvision_range', 'has_blindsight', 'has_truesight', 'has_tremorsense',
    'passive_perception',
    'trait_count', 'action_count', 'reaction_count', 'bonus_action_count',
    'legendary_action_count', 'legendary_actions_per_round',
    'total_ability_count',
    'has_multiattack', 'highest_save_dc',
    'has_spellcasting', 'spellcaster_level',
    'size_ordinal', 'has_grapple'
]

# Add condition features to Phase 3
for condition in conditions:
    phase3_features.append(f'inflicts_{condition}')

# Combine all features
feature_columns = phase1_features + phase2_features + phase3_features

print(f"📊 Total features: {len(feature_columns)}")
print(f"   - Phase 1 (Baseline): {len(phase1_features)}")
print(f"   - Phase 2 (Deviations + Scaled Abilities): {len(phase2_features)}")
print(f"     - Stat deviations: 3")
print(f"     - Scaled abilities: 5")
print(f"   - Phase 3 (Other Abilities): {len(phase3_features)}")
print(f"     - Condition features: {len(conditions)}")
print(f"\n✅ NO monster type features included")
print(f"✅ Powerful abilities now scale with baseline HP!")


In [ ]:
# Parse HP from monster data
def parse_hp(hp_str):
    if pd.isna(hp_str):
        return 0
    hp_str = str(hp_str).strip()
    match = re.match(r'(\d+)', hp_str)
    return int(match.group(1)) if match else 0

df['actual_hp'] = df['HP'].apply(parse_hp)

# Build X and y
X = df[feature_columns].fillna(0)
y = df['actual_hp']

# Remove rows with HP = 0
valid_idx = y > 0
X = X[valid_idx]
y = y[valid_idx]
df_valid = df[valid_idx].copy()

print(f"✅ Valid samples: {len(X)} monsters with HP > 0")

## Train Constrained Model with Fixed Penalties

In [ ]:
print("🤖 TRAINING CONSTRAINED MODEL")
print("=" * 60)

# User-specified constraints
# NOTE: HP baselines have been adjusted (+50% for CR ≤ 1, +20% for CR ≥ 2)
# So we need to reduce the percentage penalties proportionally
CONSTRAINTS = {
    # Stat deviations (Phase 2)
    'ac_deviation': -5.0,                           # -5 HP per point above baseline AC
    'attack_deviation': -6.0,                       # -6 HP per point above baseline attack
    'dpr_deviation': -2.5,                          # -2.5 HP per point above baseline DPR
    
    # Scaled abilities (Phase 2) - penalties as % of ADJUSTED baseline HP
    # Reduced from previous values to account for higher baselines
    'has_flying_scaled': -0.10,                     # -10% of baseline HP (was -15%)
    'has_legendary_resistance_scaled': -0.15,       # -15% of baseline HP (was -20%)
    'has_magic_resistance_scaled': -0.12,           # -12% of baseline HP (was -18%)
    'has_regeneration_scaled': -0.18,               # -18% of baseline HP (was -25%)
    'has_legendary_actions_scaled': -0.08           # -8% of baseline HP (was -12%)
}

print("📌 Fixed penalties:")
print("\nStat Deviations:")
print(f"   ac_deviation              = -5.0 HP per point")
print(f"   attack_deviation          = -6.0 HP per point")
print(f"   dpr_deviation             = -2.5 HP per point")
print("\nScaled Abilities (% of adjusted baseline HP):")
print(f"   has_flying                = -10% of baseline HP")
print(f"   has_legendary_resistance  = -15% of baseline HP")
print(f"   has_magic_resistance      = -12% of baseline HP")
print(f"   has_regeneration          = -18% of baseline HP")
print(f"   has_legendary_actions     = -8% of baseline HP")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Get indices of constrained features
constrained_indices = [feature_columns.index(feat) for feat in CONSTRAINTS.keys()]
constrained_values_scaled = []

for feat in CONSTRAINTS.keys():
    idx = feature_columns.index(feat)
    constraint_scaled = CONSTRAINTS[feat] * scaler.scale_[idx]
    constrained_values_scaled.append(constraint_scaled)

print(f"\n🔧 Constrained features: {len(constrained_indices)}")
print(f"🔓 Free features: {len(feature_columns) - len(constrained_indices)}")

# Adjust y by subtracting contribution of constrained features
y_train_adjusted = y_train.values.astype(float).copy()
y_test_adjusted = y_test.values.astype(float).copy()

for idx, val in zip(constrained_indices, constrained_values_scaled):
    y_train_adjusted -= X_train_scaled[:, idx] * val
    y_test_adjusted -= X_test_scaled[:, idx] * val

# Create mask for free features
free_mask = np.ones(len(feature_columns), dtype=bool)
free_mask[constrained_indices] = False
free_indices = np.where(free_mask)[0]

# Train on free features only
X_train_free = X_train_scaled[:, free_indices]
X_test_free = X_test_scaled[:, free_indices]

# Solve normal equations
XtX = X_train_free.T @ X_train_free
Xty = X_train_free.T @ y_train_adjusted
alpha = 1e-6
coef_free = np.linalg.solve(XtX + alpha * np.eye(len(free_indices)), Xty)

# Construct full coefficient vector
coef_full = np.zeros(len(feature_columns))
coef_full[free_indices] = coef_free
for idx, val in zip(constrained_indices, constrained_values_scaled):
    coef_full[idx] = val

# Calculate intercept
intercept = y_train.mean() - (X_train_scaled @ coef_full).mean()

# Evaluate
y_pred_train = X_train_scaled @ coef_full + intercept
y_pred_test = X_test_scaled @ coef_full + intercept

train_r2 = 1 - np.sum((y_train - y_pred_train)**2) / np.sum((y_train - y_train.mean())**2)
test_r2 = 1 - np.sum((y_test - y_pred_test)**2) / np.sum((y_test - y_test.mean())**2)
mae = np.mean(np.abs(y_test - y_pred_test))

print(f"\n📊 Model Performance:")
print(f"   Train R²: {train_r2:.4f}")
print(f"   Test R²:  {test_r2:.4f}")
print(f"   Test MAE: {mae:.2f} HP")


## Verify Constraints

In [ ]:
print("✅ VERIFYING CONSTRAINTS")
print("=" * 60)

for feat in CONSTRAINTS.keys():
    idx = feature_columns.index(feat)
    raw_coef = coef_full[idx] / scaler.scale_[idx]
    expected = CONSTRAINTS[feat]
    print(f"{feat:25s}: Expected {expected:+7.1f}, Got {raw_coef:+7.1f} ✓")

## Analyze Phase 3 Coefficients

In [ ]:
print("📈 PHASE 3 ABILITY IMPACTS")
print("=" * 60)

# Unscale coefficients
coef_unscaled = {}
for i, feat in enumerate(feature_columns):
    coef_unscaled[feat] = coef_full[i] / scaler.scale_[i]

# Display condition impacts
print(f"\n{'Condition':<20s} {'HP Impact':>12s} {'Prevalence':>12s}")
print("-" * 50)

condition_results = []
for condition in conditions:
    feature_name = f'inflicts_{condition}'
    hp_impact = coef_unscaled.get(feature_name, 0)
    count = df_valid[feature_name].sum()
    pct = count / len(df_valid) * 100
    print(f"{condition:<20s} {hp_impact:+12.1f} HP {count:4d} ({pct:4.1f}%)")
    condition_results.append({
        'condition': condition,
        'hp_impact': hp_impact,
        'count': int(count),
        'percentage': pct
    })

# Display top Phase 3 features
print(f"\n\n{'Top Phase 3 Features (by absolute impact)':<40s} {'HP Impact':>12s}")
print("-" * 60)

phase3_impacts = [(feat, coef_unscaled[feat]) for feat in phase3_features]
phase3_impacts_sorted = sorted(phase3_impacts, key=lambda x: abs(x[1]), reverse=True)

for feat, impact in phase3_impacts_sorted[:20]:
    print(f"{feat:<40s} {impact:+12.1f} HP")

## Test on Aarakocra

In [ ]:
print("🧪 TESTING ON AARAKOCRA")
print("=" * 70)

# Aarakocra stats
aarakocra_cr = 0.25
aarakocra_ac = 12
aarakocra_attack = 4
aarakocra_dpr = 4.5
aarakocra_actual_hp = 13

print(f"\n📋 Aarakocra Stats:")
print(f"   CR: {aarakocra_cr}")
print(f"   AC: {aarakocra_ac}")
print(f"   Attack: +{aarakocra_attack}")
print(f"   DPR: {aarakocra_dpr}")
print(f"   Actual HP: {aarakocra_actual_hp}")

# Get baselines
hp_base = get_baseline_hp(aarakocra_cr)
ac_base = get_baseline_ac(aarakocra_cr)
attack_base = get_baseline_attack(aarakocra_cr)
dpr_base = get_baseline_dpr(aarakocra_cr)

print(f"\n📊 CR {aarakocra_cr} Baselines:")
print(f"   HP: {hp_base:.1f}")
print(f"   AC: {ac_base:.1f}")
print(f"   Attack: +{attack_base:.1f}")
print(f"   DPR: {dpr_base:.1f}")

# Calculate deviations
ac_dev = aarakocra_ac - ac_base
attack_dev = aarakocra_attack - attack_base
dpr_dev = aarakocra_dpr - dpr_base

print(f"\n📐 Deviations from Baseline:")
print(f"   AC: {ac_dev:+.1f}")
print(f"   Attack: {attack_dev:+.1f}")
print(f"   DPR: {dpr_dev:+.1f}")

# Build feature vector
features = {feat: 0 for feat in feature_columns}
features['hp_baseline'] = hp_base
features['ac_deviation'] = ac_dev
features['attack_deviation'] = attack_dev
features['dpr_deviation'] = dpr_dev
features['speed_ground'] = 20
features['speed_fly'] = 50
features['max_speed'] = 50
features['movement_types_count'] = 2
features['has_flying_scaled'] = 1 * hp_base  # Scaled with baseline HP
features['has_legendary_resistance_scaled'] = 0
features['has_magic_resistance_scaled'] = 0
features['has_regeneration_scaled'] = 0
features['has_legendary_actions_scaled'] = 0
features['passive_perception'] = 14
features['skill_proficiency_count'] = 1
features['trait_count'] = 1
features['action_count'] = 2
features['total_ability_count'] = 3
features['size_ordinal'] = 2

# Predict
X_aarakocra = np.array([[features[feat] for feat in feature_columns]])
X_aarakocra_scaled = scaler.transform(X_aarakocra)
predicted_hp = float((X_aarakocra_scaled @ coef_full + intercept)[0])

print(f"\n🔮 PREDICTION:")
print(f"   Phase 1 (Baseline HP): {hp_base:.1f}")
print(f"   Phase 2 (AC penalty): {ac_dev * CONSTRAINTS['ac_deviation']:.1f} HP")
print(f"   Phase 2 (Attack penalty): {attack_dev * CONSTRAINTS['attack_deviation']:.1f} HP")
print(f"   Phase 2 (DPR penalty): {dpr_dev * CONSTRAINTS['dpr_deviation']:.1f} HP")
print(f"   Phase 2 (Flying penalty): {CONSTRAINTS['has_flying_scaled'] * hp_base:.1f} HP ({abs(CONSTRAINTS['has_flying_scaled'])*100:.0f}% of adjusted baseline)")
print(f"\n   Predicted HP: {predicted_hp:.1f}")
print(f"   Actual HP: {aarakocra_actual_hp}")
print(f"   Error: {predicted_hp - aarakocra_actual_hp:.1f} HP ({(predicted_hp - aarakocra_actual_hp) / aarakocra_actual_hp * 100:+.1f}%)")

if abs(predicted_hp - aarakocra_actual_hp) < 5:
    print(f"\n   ✅ Excellent prediction!")
elif abs(predicted_hp - aarakocra_actual_hp) < 10:
    print(f"\n   ✓ Good prediction")
else:
    print(f"\n   ⚠️ Needs improvement")


## Visualize Model Performance

In [ ]:
# Prediction scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test set predictions
axes[0].scatter(y_test, y_pred_test, alpha=0.5, s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual HP')
axes[0].set_ylabel('Predicted HP')
axes[0].set_title(f'Test Set Predictions (R² = {test_r2:.3f})')
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.5, s=20)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted HP')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/baseline_model_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved plot to ../data/baseline_model_performance.png")

## Save Model Artifacts

In [ ]:
print("💾 SAVING MODEL ARTIFACTS")
print("=" * 60)

# Create custom model class
class BaselineHPModel:
    """Baseline HP prediction model with CR-based baselines"""
    def __init__(self, coef, intercept, scaler, feature_columns, constraints, baseline_funcs):
        self.coef_ = coef
        self.intercept_ = intercept
        self.scaler = scaler
        self.feature_columns = feature_columns
        self.constraints = constraints
        self.baseline_funcs = baseline_funcs
        
    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return X_scaled @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - y.mean())**2)
        return 1 - ss_res / ss_tot

baseline_funcs = {
    'hp': hp_baseline_interp,
    'ac': ac_baseline_interp,
    'attack': attack_baseline_interp,
    'dpr': dpr_baseline_interp
}

model = BaselineHPModel(coef_full, intercept, scaler, feature_columns, CONSTRAINTS, baseline_funcs)

# Save model
with open('../pickled_models/baseline_hp_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✅ Saved: ../pickled_models/baseline_hp_model.pkl")

# Save scaler
with open('../pickled_models/baseline_hp_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Saved: ../pickled_models/baseline_hp_scaler.pkl")

# Save feature columns
with open('../pickled_models/baseline_hp_features.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)
print("✅ Saved: ../pickled_models/baseline_hp_features.pkl")

# Save baseline interpolation data
baseline_data = {
    'cr_values': lazy_5e['cr_numeric'].tolist(),
    'hp_baseline': lazy_5e['hp_baseline'].tolist(),
    'ac_baseline': lazy_5e['ac_baseline'].tolist(),
    'attack_baseline': lazy_5e['attack_baseline'].tolist(),
    'dpr_baseline': lazy_5e['dpr_baseline'].tolist()
}

with open('../data/baseline_lookup.json', 'w') as f:
    json.dump(baseline_data, f, indent=2)
print("✅ Saved: ../data/baseline_lookup.json")

In [ ]:
# Export to JSON for web app
# Unscale all coefficients for JSON export
coefficients_unscaled = {}
for i, feat in enumerate(feature_columns):
    coefficients_unscaled[feat] = float(coef_full[i] / scaler.scale_[i])

model_export = {
    'intercept': float(intercept),
    'coefficients': coefficients_unscaled,
    'scaler_mean': {feat: float(mean) for feat, mean in zip(feature_columns, scaler.mean_)},
    'scaler_scale': {feat: float(scale) for feat, scale in zip(feature_columns, scaler.scale_)},
    'feature_columns': feature_columns,
    'constraints': CONSTRAINTS,
    'baseline_data': baseline_data,
    'phase1_features': phase1_features,
    'phase2_features': phase2_features,
    'phase3_features': phase3_features,
    'model_type': 'baseline_hp_model',
    'description': 'Three-phase model with CR baselines'
}

with open('../monster-builder-app/model_data_baseline.json', 'w') as f:
    json.dump(model_export, f, indent=2)
print("✅ Saved: ../monster-builder-app/model_data_baseline.json")

# Save condition analysis
with open('../data/baseline_model_condition_analysis.json', 'w') as f:
    json.dump(condition_results, f, indent=2)
print("✅ Saved: ../data/baseline_model_condition_analysis.json")

## Summary

In [ ]:
print("\n" + "=" * 70)
print("🎉 BASELINE HP MODEL COMPLETE")
print("=" * 70)

print(f"\n📊 MODEL STATISTICS:")
print(f"   - Total features: {len(feature_columns)}")
print(f"   - Phase 1 (Baseline): {len(phase1_features)}")
print(f"   - Phase 2 (Deviations): {len(phase2_features)}")
print(f"   - Phase 3 (Abilities): {len(phase3_features)}")
print(f"   - Test R²: {test_r2:.4f}")
print(f"   - Test MAE: {mae:.2f} HP")

print(f"\n🔒 CONSTRAINTS (Phase 2):")
for feat, val in CONSTRAINTS.items():
    print(f"   - {feat:25s} = {val:+.1f} HP per point")

print(f"\n🔝 TOP 5 HP-INCREASING ABILITIES (Phase 3):")
sorted_conditions = sorted(condition_results, key=lambda x: x['hp_impact'], reverse=True)
for i, result in enumerate(sorted_conditions[:5], 1):
    print(f"   {i}. {result['condition']:<15s}: +{result['hp_impact']:6.1f} HP ({result['count']} monsters)")

print(f"\n🔻 TOP 5 HP-DECREASING ABILITIES (Phase 3):")
for i, result in enumerate(sorted_conditions[-5:], 1):
    print(f"   {i}. {result['condition']:<15s}: {result['hp_impact']:+6.1f} HP ({result['count']} monsters)")

print(f"\n✨ Key Improvements:")
print(f"   - Uses CR-based baselines from Lazy 5e")
print(f"   - Penalties based on deviation, not absolute values")
print(f"   - NO monster type features (cleaner model)")
print(f"   - More accurate for low-CR creatures")

print(f"\n✅ Model ready for web app integration!")